baseline

In [1]:
import pybaseball as pyb
import pybaseball.team_results
import pandas as pd
import numpy as np

# ==========================================
# THE MONKEYPATCH (FIXES PYBASEBALL'S BUG)
# ==========================================
# We rewrite the broken function inside pybaseball on the fly 
# so it safely handles the "Unknown" text without crashing.

def safe_make_numeric(data):
    num_cols = ["R", "RA", "Inn", "Rank", "Attendance"]
    for col in num_cols:
        if col in data.columns:
            # Force conversion to numbers and turn text like "Unknown" into NaN safely
            data[col] = pd.to_numeric(
                data[col].astype(str).str.replace('Unknown', '', regex=False), 
                errors='coerce'
            )
    return data

# Apply the patch to the library!
pyb.team_results.make_numeric = safe_make_numeric


# ==========================================
# YOUR ML PIPELINE
# ==========================================
YEAR = 2026 

print(f"Pulling Dodgers schedule for {YEAR}...")
dodgers_data = pyb.schedule_and_record(YEAR, 'LAD')

print(f"Pulling league pitching data for {YEAR}...")
pitching_stats = pyb.pitching_stats_bref(YEAR)

# 1. Clean the Win column to get JUST the pitcher's last name
dodgers_data['Pitcher_Last_Name'] = dodgers_data['Win'].str.extract(r'([A-Za-z\s\.\-]+)')[0].str.strip()

# 2. Extract the Last Name from the pitching stats
pitching_stats['Last_Name'] = pitching_stats['Name'].apply(lambda x: str(x).split()[-1] if pd.notnull(x) else '')

# 3. Keep only Last Name and ERA (remove duplicates)
pitcher_era_df = pitching_stats[['Last_Name', 'ERA']].drop_duplicates(subset=['Last_Name'], keep='first')

# 4. Merge on LAST NAME
model_data = pd.merge(
    dodgers_data, 
    pitcher_era_df, 
    how='left', 
    left_on='Pitcher_Last_Name', 
    right_on='Last_Name'
)

# 5. Clean target variable and drop missing data
model_data['is_win'] = model_data['W/L'].apply(lambda x: 1 if 'W' in str(x) else 0)
model_data = model_data.dropna(subset=['ERA'])

print("\nSuccess! Merged Dataset Sample:")
print(model_data[['Date', 'Opp', 'Pitcher_Last_Name', 'ERA', 'is_win']].head())

Pulling Dodgers schedule for 2026...
https://www.baseball-reference.com/teams/LAD/2026-schedule-scores.shtml


c:\Users\klzho\AppData\Local\Programs\Python\Python313\Lib\site-packages\pybaseball\team_results.py:75: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Attendance'].replace(r'^Unknown$', np.nan, regex=True, inplace = True) # make this a NaN so the column can benumeric


Pulling league pitching data for 2026...

Success! Merged Dataset Sample:
               Date  Opp Pitcher_Last_Name   ERA  is_win
0  Thursday, Mar 26  ARI          Yamamoto  2.76       1
1    Friday, Mar 27  ARI         Henriquez  3.27       1
2  Saturday, Mar 28  ARI             Klein  4.15       1
3    Monday, Mar 30  CLE           Messick  2.57       0
4   Tuesday, Mar 31  CLE            Ohtani  1.79       1


In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split


# ==========================================
# 1. FEATURE ENGINEERING
# ==========================================
model_data['is_home'] = model_data['Home_Away'].apply(lambda x: 1 if x == 'Home' else 0)

# Rolling 5-game momentum
model_data['runs_scored_last_5'] = model_data['R'].shift(1).rolling(5).mean()
model_data['runs_allowed_last_5'] = model_data['RA'].shift(1).rolling(5).mean()

features = ['is_home', 'ERA', 'runs_scored_last_5', 'runs_allowed_last_5']

# ==========================================
# 2. SEPARATE PLAYED VS. FUTURE GAMES (THE FIX)
# ==========================================
# Isolate the games that have already happened
played_games = model_data[model_data['W/L'].str.contains('W|L', na=False)].copy()
# Now it is safe to drop the games with missing ERAs (only for games already played)
played_games = played_games.dropna(subset=features)

# Isolate the future games
future_games = model_data[~model_data['W/L'].str.contains('W|L', na=False)].copy()

# Fill missing stats for future games so the model doesn't crash
# 1. Fill missing rolling runs with the most recent 5-game stretch
last_5_scored = played_games['R'].tail(5).mean()
last_5_allowed = played_games['RA'].tail(5).mean()
future_games['runs_scored_last_5'] = future_games['runs_scored_last_5'].fillna(last_5_scored)
future_games['runs_allowed_last_5'] = future_games['runs_allowed_last_5'].fillna(last_5_allowed)

# 2. Fill the missing ERA. Since the game hasn't happened, we use the average staff ERA
avg_staff_era = played_games['ERA'].mean()
future_games['ERA'] = future_games['ERA'].fillna(avg_staff_era)

# ==========================================
# 3. TRAIN AND PREDICT
# ==========================================
X_train = played_games[features]
y_train = played_games['is_win']

model = xgb.XGBClassifier(n_estimators=50, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(X_train, y_train)
print("Model successfully trained on 2026 games played to date!")

if not future_games.empty:
    X_future = future_games[features]
    
    # Predict win probabilities for the rest of the season!
    win_probabilities = model.predict_proba(X_future)[:, 1]
    future_games['Dodgers_Win_Prob'] = (win_probabilities * 100).round(1)
    
    print("\n--- UPCOMING DODGERS GAME PREDICTIONS ---")
    print(future_games[['Date', 'Opp', 'ERA', 'Dodgers_Win_Prob']].head(10).to_string(index=False))

Model successfully trained on 2026 games played to date!


In [3]:
future_games

,Date,Tm,Home_Away,Opp,W/L,R,RA,Inn,W-L,Rank,...,cLI,Streak,Orig. Scheduled,Pitcher_Last_Name,Last_Name,ERA,is_win,is_home,runs_scored_last_5,runs_allowed_last_5


In [5]:
# ==========================================
# MERGE DATA (NO DROPPING ROWS YET!)
# ==========================================
model_data = pd.merge(
    dodgers_data, 
    pitcher_era_df, 
    how='left', 
    left_on='Pitcher_Last_Name', 
    right_on='Last_Name'
)

# Create target variable (1 for Win, 0 for Loss). Future games will default to 0, which is fine!
model_data['is_win'] = model_data['W/L'].apply(lambda x: 1 if 'W' in str(x) else 0)

# ==========================================
# 1. FEATURE ENGINEERING
# ==========================================
model_data['is_home'] = model_data['Home_Away'].apply(lambda x: 1 if x == 'Home' else 0)

# Rolling 5-game momentum
model_data['runs_scored_last_5'] = model_data['R'].shift(1).rolling(5).mean()
model_data['runs_allowed_last_5'] = model_data['RA'].shift(1).rolling(5).mean()

features = ['is_home', 'ERA', 'runs_scored_last_5', 'runs_allowed_last_5']

# ==========================================
# 2. SEPARATE PLAYED VS. FUTURE GAMES
# ==========================================
# Isolate the games that have already happened (W/L column has a W or L)
played_games = model_data[model_data['W/L'].str.contains('W|L', na=False)].copy()

# NOW it is safe to drop missing data, because we are only looking at past games
played_games = played_games.dropna(subset=features)

# Isolate the future games (W/L column is blank/NaN)
future_games = model_data[~model_data['W/L'].str.contains('W|L', na=False)].copy()

# Fill missing stats for future games so the model has numbers to predict with
last_5_scored = played_games['R'].tail(5).mean()
last_5_allowed = played_games['RA'].tail(5).mean()
future_games['runs_scored_last_5'] = future_games['runs_scored_last_5'].fillna(last_5_scored)
future_games['runs_allowed_last_5'] = future_games['runs_allowed_last_5'].fillna(last_5_allowed)

avg_staff_era = played_games['ERA'].mean()
future_games['ERA'] = future_games['ERA'].fillna(avg_staff_era)

# ==========================================
# 3. TRAIN AND PREDICT
# ==========================================
X_train = played_games[features]
y_train = played_games['is_win']

model = xgb.XGBClassifier(n_estimators=50, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(X_train, y_train)
print("Model successfully trained on 2026 games played to date!")

if not future_games.empty:
    
    # ---------------------------------------------------------
    # THE FIX: INJECT REAL STARTING PITCHER ERAs FOR THIS WEEK
    # ---------------------------------------------------------
    # Let's say we looked up the Probable Pitchers for the next 3 games:
    # Aug 4 vs Cubs: Tarik Skubal (2.79 ERA)
    # Aug 5 vs Cubs: Eric Lauer (2.96 ERA)
    # Aug 7 vs D-Backs: Roki Sasaki (estimated 3.15 ERA)
    
    upcoming_eras = [2.79, 2.96, 3.15]
    
    # Loop through the first 3 future games and replace the generic average 
    # ERA with the actual scheduled pitcher's ERA
    for i in range(len(upcoming_eras)):
        # .iloc lets us select the row by its index position
        future_games.iloc[i, future_games.columns.get_loc('ERA')] = upcoming_eras[i]
        
    # ---------------------------------------------------------
    
    # Now run the prediction with the accurate pitching data!
    X_future = future_games[features]
    
    win_probabilities = model.predict_proba(X_future)[:, 1]
    future_games['Dodgers_Win_Prob'] = (win_probabilities * 100).round(1)
    
    print("\n--- UPCOMING DODGERS GAME PREDICTIONS ---")
    print(future_games[['Date', 'Opp', 'ERA', 'Dodgers_Win_Prob']].head(5).to_string(index=False))

Model successfully trained on 2026 games played to date!

--- UPCOMING DODGERS GAME PREDICTIONS ---
            Date Opp      ERA  Dodgers_Win_Prob
  Tuesday, Aug 4 CHC 2.790000         84.099998
Wednesday, Aug 5 CHC 2.960000         83.300003
   Friday, Aug 7 ARI 3.150000         83.300003
 Saturday, Aug 8 ARI 3.583846         57.400002
   Sunday, Aug 9 ARI 3.583846         57.400002


makov chain

In [13]:
import requests

# 1. Fetch Hitting and Pitching Data from the MLB API
headers = {'User-Agent': 'Mozilla/5.0'}
hit_url = "https://statsapi.mlb.com/api/v1/teams/stats?season=2026&stats=season&group=hitting&sportIds=1"
pit_url = "https://statsapi.mlb.com/api/v1/teams/stats?season=2026&stats=season&group=pitching&sportIds=1"

hit_data = requests.get(hit_url, headers=headers).json()['stats'][0]['splits']
pit_data = requests.get(pit_url, headers=headers).json()['stats'][0]['splits']

def get_rates(team_stats):
    """Helper to convert raw API stats into per-PA (or per-BF) rates."""
    # THE FIX: Hitting uses 'plateAppearances', Pitching uses 'battersFaced'
    pa = team_stats.get('plateAppearances', team_stats.get('battersFaced'))
    
    walks = team_stats['baseOnBalls'] + team_stats['hitByPitch']
    singles = team_stats['hits'] - (team_stats['doubles'] + team_stats['triples'] + team_stats['homeRuns'])
    
    return {
        '1B': singles / pa,
        '2B': team_stats['doubles'] / pa,
        '3B': team_stats['triples'] / pa,
        'HR': team_stats['homeRuns'] / pa,
        'BB': walks / pa
    }

def get_league_average_rates(api_data):
    """Sums all 30 teams to find the MLB baseline rates."""
    league_stats = {'plateAppearances': 0, 'baseOnBalls': 0, 'hitByPitch': 0, 
                    'hits': 0, 'doubles': 0, 'triples': 0, 'homeRuns': 0}
    
    for team in api_data:
        for stat in league_stats.keys():
            league_stats[stat] += team['stat'][stat]
            
    return get_rates(league_stats)

def get_matchup_probabilities(batting_team, pitching_team, hit_data, pit_data):
    # Get League Averages
    lg_rates = get_league_average_rates(hit_data)
    
    # Get Batter Rates
    bat_stats = next(t['stat'] for t in hit_data if batting_team.lower() in t['team']['name'].lower())
    bat_rates = get_rates(bat_stats)
    
    # Get Pitcher Rates (Opposing team's pitching)
    pit_stats = next(t['stat'] for t in pit_data if pitching_team.lower() in t['team']['name'].lower())
    pit_rates = get_rates(pit_stats)
    
    expected_rates = {}
    
    # Apply the Log5-style matchup formula: (Batter * Pitcher) / League
    for event in ['1B', '2B', '3B', 'HR', 'BB']:
        expected_rates[event] = (bat_rates[event] * pit_rates[event]) / lg_rates[event]
        
    # Round everything to 3 decimals
    p_single = round(expected_rates['1B'], 3)
    p_double = round(expected_rates['2B'], 3)
    p_triple = round(expected_rates['3B'], 3)
    p_hr = round(expected_rates['HR'], 3)
    p_walk = round(expected_rates['BB'], 3)
    
    # Force Out to absorb the rounding difference so it sums perfectly to 1.0
    p_out = round(1.0 - (p_single + p_double + p_triple + p_hr + p_walk), 3)
    
    return [p_out, p_single, p_double, p_triple, p_hr, p_walk]

# ==========================================
# GENERATE THE BLENDED MATCHUP ARRAYS
# ==========================================

# Dodgers Hitting vs. Cubs Pitching
dodgers_batting_array = get_matchup_probabilities('Dodgers', 'Cubs', hit_data, pit_data)

# Cubs Hitting vs. Dodgers Pitching
cubs_batting_array = get_matchup_probabilities('Cubs', 'Dodgers', hit_data, pit_data)

print("\n--- BLENDED MATCHUP PROBABILITIES ---")
print("Index order: [Out, 1B, 2B, 3B, HR, Walk]")
print(f"Dodgers (Batting) vs Cubs (Pitching): {dodgers_batting_array}")
print(f"Cubs (Batting) vs Dodgers (Pitching): {cubs_batting_array}")


--- BLENDED MATCHUP PROBABILITIES ---
Index order: [Out, 1B, 2B, 3B, HR, Walk]
Dodgers (Batting) vs Cubs (Pitching): [0.667, 0.157, 0.033, 0.002, 0.044, 0.097]
Cubs (Batting) vs Dodgers (Pitching): [0.69, 0.119, 0.039, 0.002, 0.033, 0.117]


In [ ]:
get_matchup_probabilities()

In [15]:
import numpy as np

# ==========================================
# 1. DEFINE THE STATES & TRANSITIONS
# ==========================================
# For simplicity in this example, we will look at a simplified state transition.
# A full model uses a 25x25 matrix, but the logic is identical.

# Let's define the possible outcomes of an at-bat and their probabilities for a specific team (e.g., the Dodgers)
# Probabilities: [Out, Single, Double, Triple, Home Run, Walk]

def simulate_half_inning(batting_probs):
    """
    Simulates a random walk through a single half-inning.
    Returns the total runs scored.
    """
    outs = 0
    runs = 0
    
    # Bases: [1st, 2nd, 3rd] - 0 means empty, 1 means occupied
    bases = [0, 0, 0] 
    
    # The Random Walk: Keep rolling the dice until we hit the absorbing state (3 outs)
    while outs < 3:
        # Randomly choose an event based on the team's true probabilities
        event = np.random.choice(['Out', 'Single', 'Double', 'Triple', 'HR', 'Walk'], p=batting_probs)
        
        if event == 'Out':
            outs += 1
            
        elif event == 'HR':
            # Everyone on base scores, plus the batter
            runs += sum(bases) + 1
            bases = [0, 0, 0] # Bases clear
            
        elif event == 'Triple':
            runs += sum(bases)
            bases = [0, 0, 1]
            
        elif event == 'Double':
            # Runners on 2nd and 3rd score. Runner on 1st goes to 3rd. Batter to 2nd.
            runs += bases[1] + bases[2]
            bases = [0, 1, bases[0]] 
            
        elif event == 'Single':
            # Runner on 3rd scores. Runner on 2nd scores (simplified). 
            # Runner on 1st goes to 2nd. Batter to 1st.
            runs += bases[2] + bases[1]
            bases = [1, bases[0], 0]
            
        elif event == 'Walk':
            if bases[0] == 1:
                if bases[1] == 1:
                    if bases[2] == 1:
                        runs += 1 # Bases loaded walk!
                    else:
                        bases[2] = 1 # Runner forced to 3rd
                bases[1] = 1 # Runner forced to 2nd
            bases[0] = 1 # Batter takes 1st
            
    return runs

# ==========================================
# 2. SIMULATE A FULL GAME
# ==========================================
def simulate_game(away_probs, home_probs):
    away_score = 0
    home_score = 0
    
    # 9 Innings
    for inning in range(9):
        away_score += simulate_half_inning(away_probs)
        home_score += simulate_half_inning(home_probs)
        
    # Extra innings if tied
    while away_score == home_score:
        away_score += simulate_half_inning(away_probs)
        home_score += simulate_half_inning(home_probs)
        
    return away_score, home_score

# ==========================================
# 3. MONTE CARLO SIMULATION (PREDICT THE WINNER)
# ==========================================
# A single random walk is too volatile. We run the game 10,000 times 
# to find the true mathematical probability of a win.

dodgers_batting_probs = [0.667, 0.157, 0.033, 0.002, 0.044, 0.097]
cubs_batting_probs = [0.69, 0.119, 0.039, 0.002, 0.033, 0.117] # Slightly worse than Dodgers

dodgers_wins = 0
simulations = 10000

for _ in range(simulations):
    cubs_runs, dodgers_runs = simulate_game(cubs_batting_probs, dodgers_batting_probs)
    
    if dodgers_runs > cubs_runs:
        dodgers_wins += 1

dodgers_win_probability = dodgers_wins / simulations

print(f"After {simulations} simulated random walks:")
print(f"Dodgers Win Probability: {dodgers_win_probability * 100:.1f}%")

After 10000 simulated random walks:
Dodgers Win Probability: 60.9%


predication train data grab

In [ ]:
import os
import base64
import datetime
from datetime import timedelta
import requests
import pandas as pd
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding

# ==========================================
# 1. CONFIGURATION & RSA KEYS
# ==========================================
# Secrets are read from environment variables so nothing sensitive lives in this
# notebook or the project folder.
#
#   KALSHI_KEY_ID            -> your Kalshi API Key ID (from Settings > API Keys)
#   KALSHI_PRIVATE_KEY_PATH  -> absolute path to the downloaded RSA private key
#                              (defaults to ~/.kalshi/kalshi_private_key.pem)
#
# Set them once (PowerShell, then restart the kernel/terminal):
#   setx KALSHI_KEY_ID "your-new-key-id"
#   setx KALSHI_PRIVATE_KEY_PATH "$env:USERPROFILE\.kalshi\kalshi_private_key.pem"

KALSHI_KEY_ID = os.environ.get("KALSHI_KEY_ID")

DEFAULT_KEY_PATH = os.path.join(os.path.expanduser("~"), ".kalshi", "kalshi_private_key.pem")
PRIVATE_KEY_FILE = os.environ.get("KALSHI_PRIVATE_KEY_PATH", DEFAULT_KEY_PATH)

if not KALSHI_KEY_ID:
    raise RuntimeError(
        "Environment variable KALSHI_KEY_ID is not set. "
        "Run:  setx KALSHI_KEY_ID \"your-new-key-id\"  then restart the kernel."
    )

# Date range for the dataset
START_DATE = "2026-03-20"
END_DATE = "2026-07-01"

# Base URLs
MLB_API_BASE = "https://statsapi.mlb.com/api/v1"
KALSHI_API_BASE = "https://external-api.kalshi.com"

# Load the RSA Private Key for request signing
try:
    with open(PRIVATE_KEY_FILE, "rb") as key_file:
        private_key = serialization.load_pem_private_key(key_file.read(), password=None)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find key file '{PRIVATE_KEY_FILE}'. "
        "Download your Kalshi private key and either place it there or set "
        "KALSHI_PRIVATE_KEY_PATH to its location."
    )

# Mapping MLB API full team names to Kalshi 2/3-letter team abbreviations
TEAM_TICKERS = {
    "Los Angeles Dodgers": "LAD", "New York Yankees": "NYY", "Chicago Cubs": "CHC",
    "Boston Red Sox": "BOS", "Houston Astros": "HOU", "Atlanta Braves": "ATL",
    "Philadelphia Phillies": "PHI", "Baltimore Orioles": "BAL", "Texas Rangers": "TEX",
    "Seattle Mariners": "SEA", "Toronto Blue Jays": "TOR", "Tampa Bay Rays": "TB",
    "Minnesota Twins": "MIN", "Cleveland Guardians": "CLE", "Chicago White Sox": "CWS",
    "Detroit Tigers": "DET", "Kansas City Royals": "KC", "Los Angeles Angels": "LAA",
    "Oakland Athletics": "OAK", "New York Mets": "NYM", "Washington Nationals": "WSH",
    "Miami Marlins": "MIA", "Pittsburgh Pirates": "PIT", "Cincinnati Reds": "CIN",
    "Milwaukee Brewers": "MIL", "St. Louis Cardinals": "STL", "Colorado Rockies": "COL",
    "Arizona Diamondbacks": "ARI", "San Diego Padres": "SD", "San Francisco Giants": "SF"
}


# ==========================================
# 2. AUTHENTICATION HELPER
# ==========================================
def get_kalshi_headers(method: str, path: str) -> dict:
    """
    Generates the required RSA-PSS signature and headers for Kalshi API requests.
    'path' should be the endpoint relative to domain, e.g. /trade-api/v2/markets/...
    """
    # 1. Current timestamp in milliseconds
    timestamp = str(int(datetime.datetime.now(datetime.timezone.utc).timestamp() * 1000))
    
    # 2. Strip query parameters if present for the signature payload
    path_without_query = path.split('?')[0]
    
    # 3. Form message payload: timestamp + method + path
    message = f"{timestamp}{method.upper()}{path_without_query}".encode('utf-8')
    
    # 4. Sign message with RSA-PSS SHA256
    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.DIGEST_LENGTH
        ),
        hashes.SHA256()
    )
    
    return {
        'KALSHI-ACCESS-KEY': KALSHI_KEY_ID,
        'KALSHI-ACCESS-SIGNATURE': base64.b64encode(signature).decode('utf-8'),
        'KALSHI-ACCESS-TIMESTAMP': timestamp,
        'Content-Type': 'application/json'
    }


# ==========================================
# 3. MLB DATA FETCHING
# ==========================================
def get_mlb_schedule_and_results() -> list:
    """Fetches all completed regular season games, start times, and results."""
    print(f"Fetching MLB schedule and scores from {START_DATE} to {END_DATE}...")
    url = f"{MLB_API_BASE}/schedule?sportId=1&startDate={START_DATE}&endDate={END_DATE}"
    
    response = requests.get(url)
    response.raise_for_status()
    dates = response.json().get('dates', [])
    
    games_list = []
    
    for day in dates:
        for game in day['games']:
            # 'R' = Regular Season, 'F' = Final score available
            if game.get('gameType') == 'R' and game.get('status', {}).get('statusCode') == 'F':
                game_time_utc = datetime.datetime.strptime(
                    game['gameDate'], "%Y-%m-%dT%H:%M:%SZ"
                ).replace(tzinfo=datetime.timezone.utc)
                
                away_team = game['teams']['away']['team']['name']
                home_team = game['teams']['home']['team']['name']
                
                away_score = game['teams']['away']['score']
                home_score = game['teams']['home']['score']
                winner = away_team if away_score > home_score else home_team
                
                if away_team in TEAM_TICKERS and home_team in TEAM_TICKERS:
                    games_list.append({
                        "game_date": game_time_utc.strftime("%Y-%m-%d"),
                        "start_time_utc": game_time_utc,
                        "away_team": away_team,
                        "home_team": home_team,
                        "away_score": away_score,
                        "home_score": home_score,
                        "winner": winner,
                        "ticker_away": TEAM_TICKERS[away_team],
                        "ticker_home": TEAM_TICKERS[home_team]
                    })
                    
    return games_list


# ==========================================
# 4. KALSHI HISTORICAL ODDS FETCHING
# ==========================================
def get_1_hour_before_odds(ticker: str, game_start_utc: datetime.datetime):
    """
    Queries Kalshi's candlestick endpoint, trying Live first then Historical.
    """
    start_ts = int((game_start_utc - timedelta(hours=3)).timestamp())
    end_ts = int((game_start_utc - timedelta(hours=1)).timestamp())
    
    params = {
        "period_interval": 60,
        "start_ts": start_ts,
        "end_ts": end_ts,
        "include_latest_before_start": "true" # The fix for illiquid windows
    }
    
    # Due to data partitioning, older games are moved to the /historical path
    paths_to_try = [
        f"/trade-api/v2/markets/{ticker}/candlesticks",
        f"/trade-api/v2/historical/markets/{ticker}/candlesticks"
    ]
    
    for endpoint_path in paths_to_try:
        full_url = f"{KALSHI_API_BASE}{endpoint_path}"
        headers = get_kalshi_headers("GET", endpoint_path)
        
        response = requests.get(full_url, headers=headers, params=params)
        
        if response.status_code == 200:
            candlesticks = response.json().get('candlesticks', [])
            if candlesticks:
                # Iterate backward through the candlesticks to find the last actual traded price
                for candle in reversed(candlesticks):
                    close_price = candle.get('price', {}).get('close')
                    
                    # If we find a valid price, return it immediately
                    if close_price is not None:
                        return float(close_price)
                
                # If the loop finishes and every single candle was None, return None to skip safely
                return None
                
    return None

# ==========================================
# 5. PIPELINE EXECUTION & TICKER MAPPING
# ==========================================
def fetch_all_kalshi_mlb_markets():
    """Pulls all MLB markets from BOTH Live and Historical Kalshi indexes."""
    print("Building Kalshi ticker index (Querying Live + Historical Data)...")
    market_tickers = []
    
    # We must check BOTH the live markets and the historical partitioned markets
    endpoints = [
        "/trade-api/v2/markets?series_ticker=KXMLBGAME&limit=1000",
        "/trade-api/v2/historical/markets?series_ticker=KXMLBGAME&limit=1000"
    ]
    
    for base_endpoint in endpoints:
        url = f"{KALSHI_API_BASE}{base_endpoint}"
        
        while url:
            # The RSA signature requires just the path without domain or query params
            path_for_auth = url.replace(KALSHI_API_BASE, "").split("?")[0]
            headers = get_kalshi_headers("GET", path_for_auth)
            
            response = requests.get(url, headers=headers)
            
            if response.status_code != 200:
                break
                
            data = response.json()
            for market in data.get('markets', []):
                market_tickers.append(market['ticker'])
                
            cursor = data.get('cursor')
            if cursor:
                url = f"{KALSHI_API_BASE}{base_endpoint}&cursor={cursor}"
            else:
                url = None
                
    return market_tickers

def find_matching_kalshi_ticker(game, kalshi_tickers):
    """Matches an MLB API game to the exact Kalshi market ticker."""
    date_str = game['start_time_utc'].strftime("%y%b%d").upper()
    away = game['ticker_away']
    home = game['ticker_home']
    
    # We want the "Yes" contract for the Away team. 
    prefix = f"KXMLBGAME-{date_str}"
    
    for ticker in kalshi_tickers:
        if ticker.startswith(prefix) and away in ticker and home in ticker and ticker.endswith(f"-{away}"):
            return ticker
            
    return None

def build_dataset():
    games = get_mlb_schedule_and_results()
    dataset = []
    
    kalshi_tickers = fetch_all_kalshi_mlb_markets()
    print(f"\nFound {len(games)} MLB games and {len(kalshi_tickers)} Kalshi markets. Starting extraction...\n")
    
    for i, game in enumerate(games, 1):
        ticker = find_matching_kalshi_ticker(game, kalshi_tickers)
        
        if not ticker:
            print(f"[{i}/{len(games)}] Skipped: No matching Kalshi market found for {game['away_team']} @ {game['home_team']} on {game['game_date']}")
            continue
            
        away_prob = get_1_hour_before_odds(ticker, game['start_time_utc'])
        
        if away_prob is not None:
            game['away_implied_win_prob'] = round(away_prob, 3)
            game['home_implied_win_prob'] = round(1.0 - away_prob, 3)
            dataset.append(game)
            print(f"[{i}/{len(games)}] Success: {ticker} | Away Win Prob: {game['away_implied_win_prob']}")
        else:
            print(f"[{i}/{len(games)}] Skipped: {ticker} (No market history found)")
            
    df = pd.DataFrame(dataset)
    if not df.empty:
        df = df.drop(columns=['ticker_away', 'ticker_home'])
        output_file = "MLB_2026_First_Half_Odds_And_Results.csv"
        df.to_csv(output_file, index=False)
        print(f"\nDataset successfully saved to '{output_file}' with {len(df)} games.")
    else:
        print("\nNo matching game odds were found.")


In [39]:
build_dataset()

Fetching MLB schedule and scores from 2026-03-20 to 2026-07-01...
Building Kalshi ticker index (Querying Live + Historical Data)...

Found 1208 MLB games and 8402 Kalshi markets. Starting extraction...

[1/1208] Skipped: No matching Kalshi market found for New York Yankees @ San Francisco Giants on 2026-03-26
[2/1208] Success: KXMLBGAME-26MAR261315PITNYM-PIT | Away Win Prob: 0.49
[3/1208] Success: KXMLBGAME-26MAR261410CWSMIL-CWS | Away Win Prob: 0.37
[4/1208] Success: KXMLBGAME-26MAR261420WSHCHC-WSH | Away Win Prob: 0.34
[5/1208] Success: KXMLBGAME-26MAR261505MINBAL-MIN | Away Win Prob: 0.45
[6/1208] Success: KXMLBGAME-26MAR261610BOSCIN-BOS | Away Win Prob: 0.6
[7/1208] Success: KXMLBGAME-26MAR261610LAAHOU-LAA | Away Win Prob: 0.39
[8/1208] Success: KXMLBGAME-26MAR261610DETSD-DET | Away Win Prob: 0.54
[9/1208] Success: KXMLBGAME-26MAR261615TBSTL-TB | Away Win Prob: 0.56
[10/1208] Success: KXMLBGAME-26MAR261615TEXPHI-TEX | Away Win Prob: 0.41
[11/1208] Skipped: No matching Kalshi market

In [4]:
import pandas as pd

data = pd.read_csv("MLB_2026_First_Half_Odds_And_Results.csv")

In [5]:
data

,game_date,start_time_utc,away_team,home_team,away_score,home_score,winner,away_implied_win_prob,home_implied_win_prob
0,2026-03-26,2026-03-26 17:15:00+00:00,Pittsburgh Pirates,New York Mets,7,11,New York Mets,0.49,0.51
1,2026-03-26,2026-03-26 18:10:00+00:00,Chicago White Sox,Milwaukee Brewers,2,14,Milwaukee Brewers,0.37,0.63
2,2026-03-26,2026-03-26 18:20:00+00:00,Washington Nationals,Chicago Cubs,10,4,Washington Nationals,0.34,0.66
3,2026-03-26,2026-03-26 19:05:00+00:00,Minnesota Twins,Baltimore Orioles,1,2,Baltimore Orioles,0.45,0.55
4,2026-03-26,2026-03-26 20:10:00+00:00,Boston Red Sox,Cincinnati Reds,3,0,Boston Red Sox,0.60,0.40
...,...,...,...,...,...,...,...,...,...
821,2026-06-07,2026-06-07 18:15:00+00:00,Cincinnati Reds,St. Louis Cardinals,3,5,St. Louis Cardinals,0.44,0.56
822,2026-06-07,2026-06-07 18:35:00+00:00,Cleveland Guardians,Texas Rangers,0,10,Texas Rangers,0.44,0.56
823,2026-06-07,2026-06-07 19:10:00+00:00,Milwaukee Brewers,Colorado Rockies,12,4,Milwaukee Brewers,0.63,0.37
824,2026-06-07,2026-06-07 20:10:00+00:00,Los Angeles Angels,Los Angeles Dodgers,13,5,Los Angeles Angels,0.31,0.69


In [44]:
def get_matchup_probabilities(batting_team, pitching_team, hit_data, pit_data):
    # Get League Averages
    lg_rates = get_league_average_rates(hit_data)
    
    # Get Batter Rates (Exact match on full team name)
    bat_stats = next(t['stat'] for t in hit_data if batting_team.lower() == t['team']['name'].lower())
    bat_rates = get_rates(bat_stats)
    
    # Get Pitcher Rates (Exact match on full team name)
    pit_stats = next(t['stat'] for t in pit_data if pitching_team.lower() == t['team']['name'].lower())
    pit_rates = get_rates(pit_stats)
    
    expected_rates = {}
    
    # Apply the Log5-style matchup formula: (Batter * Pitcher) / League
    for event in ['1B', '2B', '3B', 'HR', 'BB']:
        expected_rates[event] = (bat_rates[event] * pit_rates[event]) / lg_rates[event]
        
    # Round everything to 3 decimals
    p_single = round(expected_rates['1B'], 3)
    p_double = round(expected_rates['2B'], 3)
    p_triple = round(expected_rates['3B'], 3)
    p_hr = round(expected_rates['HR'], 3)
    p_walk = round(expected_rates['BB'], 3)
    
    # Force Out to absorb the rounding difference so it sums perfectly to 1.0
    p_out = round(1.0 - (p_single + p_double + p_triple + p_hr + p_walk), 3)
    
    return [p_out, p_single, p_double, p_triple, p_hr, p_walk]

In [1]:
#batting
batting_until_july = "https://statsapi.mlb.com/api/v1/teams/stats?season=2026&stats=byDateRange&startDate=2026-03-01&endDate=2026-07-01&group=hitting&sportIds=1&gameType=R"
#pitching
pitching_until_july = "https://statsapi.mlb.com/api/v1/teams/stats?season=2026&stats=byDateRange&startDate=2026-03-01&endDate=2026-07-01&group=pitching&sportIds=1&gameType=R"

In [46]:
for i in data.head(10).itertuples():
    away_team = i.away_team
    home_team = i.home_team
    
    dodgers_batting_array = get_matchup_probabilities(away_team, home_team, hit_data, pit_data)
    cubs_batting_array = get_matchup_probabilities(home_team, away_team, hit_data, pit_data)
    
    print(f"\nMatchup Probabilities for {away_team} vs {home_team}:")
    print(f"{away_team} (Batting) vs {home_team} (Pitching): {dodgers_batting_array}")
    print(f"{home_team} (Batting) vs {away_team} (Pitching): {cubs_batting_array}")


Matchup Probabilities for Pittsburgh Pirates vs New York Mets:
Pittsburgh Pirates (Batting) vs New York Mets (Pitching): [0.671, 0.144, 0.044, 0.003, 0.029, 0.109]
New York Mets (Batting) vs Pittsburgh Pirates (Pitching): [0.696, 0.134, 0.039, 0.002, 0.027, 0.102]

Matchup Probabilities for Chicago White Sox vs Milwaukee Brewers:
Chicago White Sox (Batting) vs Milwaukee Brewers (Pitching): [0.71, 0.122, 0.037, 0.002, 0.03, 0.099]
Milwaukee Brewers (Batting) vs Chicago White Sox (Pitching): [0.663, 0.142, 0.044, 0.005, 0.021, 0.125]

Matchup Probabilities for Washington Nationals vs Chicago Cubs:
Washington Nationals (Batting) vs Chicago Cubs (Pitching): [0.681, 0.14, 0.034, 0.003, 0.049, 0.093]
Chicago Cubs (Batting) vs Washington Nationals (Pitching): [0.647, 0.147, 0.041, 0.002, 0.036, 0.127]

Matchup Probabilities for Minnesota Twins vs Baltimore Orioles:
Minnesota Twins (Batting) vs Baltimore Orioles (Pitching): [0.679, 0.144, 0.049, 0.005, 0.029, 0.094]
Baltimore Orioles (Batting

In [6]:
import requests

def get_historical_last_n_games_url(team_id, target_date, stat_group="hitting", n=5, season="2026"):
    # 1. Fetch the schedule for the team from Opening Day up to the target date
    schedule_url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&teamId={team_id}&startDate={season}-03-01&endDate={target_date}&gameType=R"
    
    response = requests.get(schedule_url, headers={'User-Agent': 'Mozilla/5.0'}).json()
    
    played_dates = []
    
    # 2. Extract dates for games that were actually completed
    if 'dates' in response:
        for date_obj in response['dates']:
            # Check the status of the first game on that date
            game_status = date_obj['games'][0]['status']['statusCode']
            # 'F' = Final, 'C' = Completed Early, 'O' = Game Over
            if game_status in ['F', 'C', 'O']: 
                played_dates.append(date_obj['date'])
                
    # 3. Make sure they actually played at least N games
    if len(played_dates) < n:
        print(f"Team {team_id} played fewer than {n} games before {target_date}.")
        return None
        
    # 4. Grab the calendar dates for the last N games
    last_n_dates = played_dates[-n:]
    start_date = last_n_dates[0]
    end_date = last_n_dates[-1] 
    
    # 5. Construct and return your final stats URL using byDateRange
    stats_url = (f"https://statsapi.mlb.com/api/v1/teams/stats?season={season}"
                 f"&stats=byDateRange&startDate={start_date}&endDate={end_date}"
                 f"&group={stat_group}&sportIds=1&gameType=R&teamId={team_id}")
    
    print(f"Found {n} games between {start_date} and {end_date}.")
    return stats_url

# ==========================================
# EXAMPLE USAGE
# ==========================================

# Let's get the Dodgers (Team ID 119) Hitting stats for their 5 games prior to July 1st, 2026
target_date = "2026-07-01"
dodgers_id = 119

dodgers_last_5_hit_url = get_historical_last_n_games_url(dodgers_id, target_date, stat_group="hitting", n=5)

if dodgers_last_5_hit_url:
    # Now you can request the actual stats!
    headers = {'User-Agent': 'Mozilla/5.0'}
    stats_response = requests.get(dodgers_last_5_hit_url, headers=headers).json()
    
    # Print the raw data
    print("\nRaw API Data for this 5-game stretch:")
    print(stats_response['stats'][0]['splits'][0]['stat'])

Found 5 games between 2026-06-27 and 2026-07-01.

Raw API Data for this 5-game stretch:
{'gamesPlayed': 5, 'groundOuts': 40, 'airOuts': 50, 'runs': 33, 'doubles': 14, 'triples': 4, 'homeRuns': 6, 'strikeOuts': 37, 'baseOnBalls': 13, 'intentionalWalks': 0, 'hits': 58, 'hitByPitch': 2, 'avg': '.315', 'atBats': 184, 'obp': '.367', 'slg': '.533', 'ops': '.900', 'caughtStealing': 1, 'stolenBases': 4, 'stolenBasePercentage': '.800', 'caughtStealingPercentage': '.200', 'groundIntoDoublePlay': 7, 'numberOfPitches': 726, 'plateAppearances': 200, 'totalBases': 98, 'rbi': 33, 'leftOnBase': 32, 'sacBunts': 1, 'sacFlies': 0, 'babip': '.369', 'groundOutsToAirouts': '0.80', 'atBatsPerHomeRun': '30.67'}


In [7]:
def convert_raw_to_rates(team_stats):
    """
    Takes raw MLB API stats dictionary and returns a list of rates per PA:
    [Out, 1B, 2B, 3B, HR, Walk]
    """
    # Use plate appearances for hitting
    pa = team_stats.get('plateAppearances')
    
    # Safety check in case a team has 0 plate appearances in the timeframe
    if not pa or pa == 0:
        return [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        
    walks = team_stats.get('baseOnBalls', 0) + team_stats.get('hitByPitch', 0)
    doubles = team_stats.get('doubles', 0)
    triples = team_stats.get('triples', 0)
    home_runs = team_stats.get('homeRuns', 0)
    
    # Singles = Total Hits minus extra base hits
    singles = team_stats.get('hits', 0) - (doubles + triples + home_runs)
    
    # Calculate probabilities per Plate Appearance
    p_single = round(singles / pa, 3)
    p_double = round(doubles / pa, 3)
    p_triple = round(triples / pa, 3)
    p_hr = round(home_runs / pa, 3)
    p_walk = round(walks / pa, 3)
    
    # Force Out to absorb the remainder so the total sums to exactly 1.0
    p_out = round(1.0 - (p_single + p_double + p_triple + p_hr + p_walk), 3)
    
    return [p_out, p_single, p_double, p_triple, p_hr, p_walk]

In [ ]:
f

# Evaluation harness â€” the baseline every model must beat

We predict **P(home team wins)** for each game and score it against the realized
outcome. The Kalshi market's `home_implied_win_prob` is the benchmark: any model
worth keeping has to beat the market's Brier / log loss **on the same games**.

Metrics live in `metrics.py` (`evaluate`, `compare`). Reference points for a
50/50 coin: Brier `0.2500`, log loss `0.6931`.


In [ ]:
import pandas as pd
import numpy as np
from metrics import evaluate, compare

games = pd.read_csv("MLB_2026_First_Half_Odds_And_Results.csv")

# Realized outcome: did the home team win?
games["home_win"] = (games["home_score"] > games["away_score"]).astype(int)

# ---- data sanity checks ----
assert games[["home_implied_win_prob", "away_implied_win_prob"]].notna().all().all()
assert (games["home_score"] != games["away_score"]).all(), "unexpected tie game"
psum = games["away_implied_win_prob"] + games["home_implied_win_prob"]
print(f"implied prob pairs sum to [{psum.min():.3f}, {psum.max():.3f}] -> no vig to remove")

# A few market lines sit at 0.01 / 0.99 -- almost certainly stale / illiquid
# Kalshi markets picked up by the 'latest before start' fallback. They dominate
# log loss, so we flag them and also report the market's score without them.
liquid = games["home_implied_win_prob"].between(0.10, 0.90, inclusive="neither")
print(f"{(~liquid).sum()} games have a market prob <=0.10 or >=0.90 (suspect lines)\n")

# ==========================================
# BASELINES
# ==========================================
results = [
    evaluate(games["home_win"], games["home_implied_win_prob"], name="Kalshi market"),
    evaluate(games["home_win"], np.full(len(games), 0.5),
             name="always 0.50", show_calibration=False),
    evaluate(games["home_win"], np.full(len(games), games["home_win"].mean()),
             name="always home base-rate (in-sample)", show_calibration=False),
]

# Robustness: market score on the 823 non-suspect games
evaluate(games.loc[liquid, "home_win"], games.loc[liquid, "home_implied_win_prob"],
         name=f"Kalshi market (excl. suspect lines, n={liquid.sum()})",
         show_calibration=False)

print("LEADERBOARD (sorted by log loss)")
print(compare(results).to_string(index=False))


### What the baseline numbers say

| model | Brier | log loss | acc | ECE |
|---|---|---|---|---|
| always home base-rate (0.53) | 0.2491 | 0.6913 | 53.0% | 0.000 |
| always 0.50 | 0.2500 | 0.6931 | 53.0% | 0.030 |
| **Kalshi market** | **0.2485** | 0.6959 | **55.9%** | 0.022 |
| Kalshi market, excl. 3 suspect lines | 0.2470 | 0.6873 | 56.0% | 0.020 |

- **The bar is low and the ceiling is low.** MLB games are close to coin flips. The
  market's Brier (0.2485) beats a coin (0.2500) by ~0.0015. That is roughly the
  entire edge available â€” a "good" model here lands around Brier 0.246â€“0.247.
- **The market loses to a constant on log loss** (0.696 vs 0.691) purely because of
  3 lines at 0.01 / 0.99 that went the wrong way. Drop those and it beats every
  constant (0.6873). Treat `home_implied_win_prob âˆ‰ (0.10, 0.90)` as bad data:
  filter them, or winsorize all probs to ~[0.03, 0.97] before scoring.
- **Where the real signal is:** accuracy 56% vs 53% base rate, and the middle
  calibration bins (which hold ~92% of games) track well â€” predicted 0.452 â†’ actual
  0.463, predicted 0.547 â†’ actual 0.560. The market ranks games correctly; it just
  can't be very confident.
- **Metric to optimize:** Brier first (robust to the extreme-line noise), log loss
  second on the filtered set, and watch ECE for calibration. Beating **Brier 0.2470**
  (market, clean games) is the target.


# Model 1 â€” Monte Carlo inning sim vs. the market

The notebook's inning simulator, packaged in `montecarlo.py` so the harness can
score it. Per game: team season rates â†’ Log5 event probabilities per side â†’
simulated half-inning run distribution â†’ 9-fold convolution to a game-score
distribution â†’ exact P(home win) with extra innings folded in. Base-running rules
are the same crude ones as the original sim.

**Caveat:** `fetch_team_tables(2026)` with no date range uses **full-season**
stats, so an April game is scored partly from its own result. The numbers below
are therefore optimistic â€” the market only knew data up to ~1 h before each game.
A fair test needs per-game as-of-date rates (next step).


In [ ]:
import montecarlo as mc

# `games` and `results` come from the evaluation-harness cell above.

# 1. team rate tables (full season -- see caveat above)
tables = mc.fetch_team_tables(2026)

# 2. P(home win) for every game; cached per matchup, ~20s at n=60k
mc_prob = mc.predict_games(games, tables, n=60_000, seed=7)
print("MC home-win prob:  min %.3f  mean %.3f  max %.3f  (home teams actually win %.3f)\n"
      % (mc_prob.min(), mc_prob.mean(), mc_prob.max(), games["home_win"].mean()))

# 3. score it on the same games, next to the market
results.append(evaluate(games["home_win"], mc_prob,
                        name="Monte Carlo (season stats, lookahead)"))

print("LEADERBOARD (sorted by log loss)")
print(compare(results).to_string(index=False))


### What Model 1 says

| model | Brier | log loss | acc | ECE |
|---|---|---|---|---|
| **Monte Carlo** (season stats, lookahead) | **0.2467** | **0.6864** | 52.2% | 0.053 |
| always base-rate (0.53) | 0.2491 | 0.6913 | 53.0% | 0.000 |
| Kalshi market | 0.2485 | 0.6959 | 55.9% | 0.022 |

Read this carefully â€” it's not the win it looks like:

- **The Brier edge is mostly free.** The sim uses full-season stats, so it has
  peeked at the games it's scoring. And the market's raw log loss is dragged down
  by the 3 junk lines; the clean-market Brier is **0.2470**, right on top of the
  sim.
- **It can't rank games.** Accuracy 52% â€” *worse* than guessing "home" every time
  (53%). Predictions never leave `~[0.28, 0.73]`. The market's 55.9% comes from
  knowing starting pitchers, lineups and injuries â€” none of which a team-aggregate
  sim sees.
- **No home-field advantage.** Mean prediction â‰ˆ 0.50 vs actual home-win rate 0.53,
  and ECE 0.053 (worst of the three). The original sim gives the home team zero
  structural edge â€” a ~1.03Ã— bump to home offense (or a small additive run term)
  would remove the bias.

**Next:** (a) as-of-date rates so the comparison is honest, (b) a home-field term,
(c) fold in the probable starting pitcher instead of team-average pitching.


# Model 1b â€” adding starter ERA and recent-form windows

Three new knobs in `montecarlo.predict_games` (all optional):

| knob | what it does |
|---|---|
| `window=5` / `10` | trailing-N-game team rates as of each game date, instead of full season (needs `logs` from `fetch_team_game_logs`) |
| `shrink_pa=N` | regress a short window toward the season rate with `N` plate appearances of prior â€” kills small-sample noise |
| `starters=â€¦`, `starter_strength` | scale the opposing offense's reach rates by the probable starter's ERA / staff ERA; `strength` (0â€“1) damps it |

`fetch_team_game_logs` is ~60 API calls, `fetch_probable_starter_eras` is 2.
The cell below is the slow one in the notebook (~90 s).


In [ ]:
# needs: games, results, tables (from the Model 1 cells), evaluate, compare

logs = mc.fetch_team_game_logs(2026)
starters = mc.fetch_probable_starter_eras("2026-03-20", "2026-07-01", 2026)
both = sum(1 for _, r in games.iterrows()
           if all(starters.get((r.game_date, r.away_team, r.home_team), {}).get(k)
                  for k in ("away_era", "home_era")))
print(f"probable-starter ERA available for {both}/{len(games)} games\n")

configs = {
    "MC season (baseline)":            dict(),
    "MC + starter ERA (strength .30)": dict(starters=starters, starter_strength=0.30),
    "MC last-10, no shrink":           dict(logs=logs, window=10),
    "MC last-10, shrink 800":          dict(logs=logs, window=10, shrink_pa=800),
    "MC last-10 sh800 + starter .30":  dict(logs=logs, window=10, shrink_pa=800,
                                           starters=starters, starter_strength=0.30),
}

model1b = [evaluate(games["home_win"], games["home_implied_win_prob"],
                    name="Kalshi market", show_calibration=False)]
for name, kw in configs.items():
    p = mc.predict_games(games, tables, n=50_000, seed=7, **kw)
    model1b.append(evaluate(games["home_win"], p, name=name, show_calibration=False))

print("LEADERBOARD (sorted by log loss)")
print(compare(model1b).to_string(index=False))


### What Model 1b says

| config | Brier | log loss | acc | ECE |
|---|---|---|---|---|
| **MC + starter ERA (strength .30)** | **0.2407** | **0.6741** | 57.0% | 0.038 |
| MC last-10, shrink 800, + starter .30 | 0.2419 | 0.6763 | 56.8% | 0.065 |
| MC season (baseline) | 0.2470 | 0.6869 | 53.1% | 0.040 |
| MC last-10, shrink 800 | 0.2482 | 0.6895 | 54.2% | 0.049 |
| Kalshi market | 0.2485 | 0.6959 | 55.9% | 0.022 |
| MC last-10, no shrink | 0.2728 | 0.7573 | 52.2% | 0.136 |

- **Starter ERA is the real add.** It moves Brier 0.247 â†’ 0.241, log loss 0.687 â†’
  0.674, accuracy 53% â†’ 57%. This is the first config that beats the market on
  Brier *and* log loss and matches it on accuracy. Sweet spot `starter_strength`
  â‰ˆ 0.2â€“0.3; at 0.45+ it over-shoots and calibration degrades again.
- **Recent-form windows don't help.** Raw last-10 is far worse (Brier 0.273, wildly
  overconfident). Shrunk hard enough to be safe (`shrink_pa=800` â‰ˆ 20 games of
  prior) it just collapses back onto the season baseline â€” the window contributes
  ~nothing. Stacked on starter ERA it slightly *hurts* (0.2419 vs 0.2407). Drop it.
- **Still lookahead.** Both the team rates and the starter ERA are full-season. The
  starter-ERA gain is probably mostly genuine â€” it's ranking information, and one
  start barely moves a season ERA â€” but confirm it with as-of-date pitcher stats
  before trusting the margin.

**Keep:** starter ERA at strength â‰ˆ 0.25. **Drop:** rolling windows.
**Next:** as-of-date team + pitcher rates; a home-field term; then blend vs market.


# Model 1c â€” today's batting order + starter rest

Rebuild each team's offense from the **9 hitters actually in today's lineup**
(`fetch_lineups`, known ~3 h before first pitch so not lookahead), slot-weighted,
each hitter's rate regressed toward the league-average hitter with
`fetch_player_hit_rates(shrink_pa=100)`. Injuries and rest-day sits fall out for
free â€” an IL'd star simply isn't in the lineup.

Also `fetch_starter_rest` â†’ days between a probable starter's last outing and this
game, folded into the pitcher scale via `rest_scale`.

`fetch_starter_rest` is slow (~130 s, one game-log call per starter). Everything
else is 3 calls total.


In [ ]:
# needs: games, tables, evaluate, compare, mc  (from the Model 1 / 1b cells)
S, E = "2026-03-20", "2026-06-10"

player_rates = mc.fetch_player_hit_rates(2026, shrink_pa=100)   # 1 call
lineups      = mc.fetch_lineups(S, E)                           # 1 call
rest         = mc.fetch_starter_rest(S, E, 2026)                # ~130 s

lu_cov = sum((r.game_date, r.away_team, r.home_team) in lineups
             for r in games.itertuples())
print(f"lineup coverage {lu_cov}/{len(games)}\n")

cfgs = {
    "MC + starter ERA .30 (Model 1b best)": dict(starters=starters, starter_strength=.30),
    "MC + lineups":                          dict(lineups=lineups, player_rates=player_rates),
    "MC + lineups + starter ERA":            dict(lineups=lineups, player_rates=player_rates,
                                                 starters=starters, starter_strength=.30),
    "MC + lineups + starter + rest":         dict(lineups=lineups, player_rates=player_rates,
                                                 starters=starters, starter_strength=.30,
                                                 rest=rest, rest_strength=1.0),
}
model1c = [evaluate(games["home_win"], games["home_implied_win_prob"],
                    name="Kalshi market", show_calibration=False)]
for name, kw in cfgs.items():
    p = mc.predict_games(games, tables, n=50_000, seed=7, **kw)
    model1c.append(evaluate(games["home_win"], p, name=name, show_calibration=False))

print("LEADERBOARD (sorted by log loss)")
print(compare(model1c).to_string(index=False))


### What Model 1c says

| config | Brier | log loss | acc | ECE |
|---|---|---|---|---|
| **MC + lineups + starter ERA** | **0.2395** | **0.6711** | **57.9%** | 0.031 |
| MC + lineups + starter + rest | 0.2394 | 0.6709 | 57.0% | 0.044 |
| MC + starter ERA .30 (1b best) | 0.2406 | 0.6739 | 56.7% | 0.038 |
| MC + lineups | 0.2452 | 0.6833 | 54.7% | 0.032 |
| MC season baseline | 0.2471 | 0.6873 | 51.9% | 0.054 |
| Kalshi market | 0.2485 | 0.6959 | 55.9% | 0.023 |

- **Lineups + starter ERA is the best model so far, and it beats the market on
  every metric** â€” Brier 0.2395 vs 0.2485, log loss 0.671 vs 0.696, accuracy
  57.9% vs 55.9%. Lineup coverage is 826/826.
- **Lineups add a real but small amount.** Alone: Brier 0.2471 â†’ 0.2452, accuracy
  52% â†’ 55%, and ECE drops to 0.032 (best calibration of any MC config). On top of
  starter ERA they stack cleanly for another ~0.001 Brier and ~1 pp accuracy.
- **Rest days do essentially nothing** â€” Brier unchanged, accuracy slightly *down*,
  calibration slightly worse. Genuine short-rest starts are rare and teams avoid
  them, so there's little to gain. Leave `rest_strength=0` unless you reshape the
  penalty curve. (Skipping `fetch_starter_rest` also saves the ~130 s fetch.)
- **Still lookahead** in the player season rates and the season starter ERA. Lineup
  *identity* is clean. The honest out-of-sample margin over the market is smaller
  than the table shows â€” pin it down with as-of-date player/pitcher stats.

**Current best recipe:** `predict_games(games, tables, lineups=lineups,
player_rates=player_rates, starters=starters, starter_strength=0.30)`.


# Model 1d — betting backtest vs. the Kalshi market

`backtest.py` buys whichever side the model values above its Kalshi price by
more than `edge_threshold`. Per contract: win → +(1−price), lose → −price,
minus Kalshi's ≈ 0.07·p·(1−p) entry fee. Flat = 1 contract/bet; Kelly =
fractional-Kelly share of the running bankroll.

Sanity check: **betting the market favorite every game returns −4.5%** — pure
fee drag on a fair market. Any real edge has to clear that.


In [ ]:
import backtest as bt

# model_prob for the current best recipe (needs lineups, player_rates, starters from 1c/1b)
best = mc.predict_games(games, tables, lineups=lineups, player_rates=player_rates,
                        starters=starters, starter_strength=0.30, n=60_000, seed=7)

thresholds = [0.00, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10]
print('FLAT stake, fees ON')
print(bt.threshold_sweep(games, best, thresholds, stake='flat', apply_fee=True).to_string(index=False))
print('
FLAT stake, fees OFF')
print(bt.threshold_sweep(games, best, thresholds, stake='flat', apply_fee=False).to_string(index=False))

r = bt.run_backtest(games, best, edge_threshold=0.04, stake='kelly',
                    bankroll=1000, kelly_fraction=0.25, kelly_cap=0.05, apply_fee=True)
print(f"
Kelly 0.25x, thr 0.04:  {r['n_bets']} bets, win {r['win_rate']:.1%}, "
      f"$1000 -> ${r['final_bankroll']:,.0f} ({r['return_pct']:+.0f}%), maxDD {r['max_drawdown_pct']:.0f}%")


### What the backtest says — and why you should not trust the size

| config | flat ROI, fees on (thr 0.03–0.05) | win rate | Kelly $1k → |
|---|---|---|---|
| MC season | +1.5% to +6.8% | ~50% | — |
| MC + starter ERA | +13% to +15% | 57–60% | large |
| **MC + lineups + starter ERA** | **+17% to +19%** | 59–61% | $1k → ~$100k |

The framework is sound — the favorite-betting baseline loses exactly the fee
(−4.5%), so the harness isn't flattering anything. But the model numbers are
**inflated** and are not evidence of a profitable strategy yet:

1. **Lookahead dominates.** The model uses full-season 2026 player rates, starter
   ERA and team pitching — it has partly seen the outcomes it's betting on. A
   +15% ROI at 58% win rate is the classic signature of a peeked model. Note the
   *MC-season* config, which leans least on leaked per-game stats, only makes
   +1.5% to +7% — that's closer to the honest ceiling, and it's within noise for
   826 bets.
2. **Frictionless fills.** One price per game, unlimited size, no slippage. Kalshi
   MLB books are thin; a 5%-of-bankroll Kelly bet ($4k on an $80k roll) would move
   the price hard against you. The $1k → $100k curve is a compounding artifact
   with a 36–40% drawdown — with a realistic (smaller, maybe zero) edge it goes
   sideways.
3. **In-sample.** ROI is computed on the same 826 games you'd tune the threshold
   on. It's flat-ish across thresholds (mildly reassuring) but still not a
   held-out test.
4. **Half a season.** 826 games. Wide error bars on any of these.

**Before believing there's money here:** (a) every input as-of game date, (b) a
fill model — cap stake per game, fill through the book not at one price, (c)
train/tune on Mar–May, report on Jun+ only, (d) then look at flat ROI after fees,
not the Kelly headline.


# Model 1e — true out-of-sample test (data ≤ Jul 1 → predict August)

Everything above trained and scored on the same Mar–Jun games with full-season
stats, so the model had partly seen its own answers. This freezes every rate
input at **July 1** and predicts **August** games it has never touched.

* team / hitter / starter rates: `byDateRange` Mar 1 → Jul 1 only
* August lineups used as-is (who's playing is known at game time; not leakage)
* no Kalshi odds for August in the CSV, so this is model quality only, not P&L


In [ ]:
CUT_S, CUT_E = '2026-03-01', '2026-07-01'      # data we're allowed to use
TEST_S, TEST_E = '2026-08-01', '2026-08-27'    # games we predict

test = mc.fetch_results(TEST_S, TEST_E)
print(f'{len(test)} August games, home-win rate {test.home_win.mean():.3f}')

tab_cut = mc.fetch_team_tables(2026, CUT_S, CUT_E)
pr_cut  = mc.fetch_player_hit_rates(2026, shrink_pa=100, start_date=CUT_S, end_date=CUT_E)
st_cut  = mc.fetch_probable_starter_eras(TEST_S, TEST_E, 2026, era_start=CUT_S, era_end=CUT_E)
lu_aug  = mc.fetch_lineups(TEST_S, TEST_E)

oos = [
    evaluate(test.home_win, np.full(len(test), 0.5), name='always 0.50', show_calibration=False),
    evaluate(test.home_win, np.full(len(test), test.home_win.mean()),
             name='always base-rate', show_calibration=False),
]
cfgs = {
    'OOS team-only':       dict(),
    'OOS + starter ERA':   dict(starters=st_cut, starter_strength=0.30),
    'OOS + lineups + SP':  dict(lineups=lu_aug, player_rates=pr_cut,
                                starters=st_cut, starter_strength=0.30),
}
for name, kw in cfgs.items():
    p = mc.predict_games(test, tab_cut, n=60_000, seed=7, **kw)
    oos.append(evaluate(test.home_win, p, name=name, show_calibration=False))

# for contrast: the same config with full-season (lookahead) stats
tab_f = mc.fetch_team_tables(2026)
pr_f  = mc.fetch_player_hit_rates(2026, shrink_pa=100)
st_f  = mc.fetch_probable_starter_eras(TEST_S, TEST_E, 2026)
pL = mc.predict_games(test, tab_f, lineups=lu_aug, player_rates=pr_f,
                      starters=st_f, starter_strength=0.30, n=60_000, seed=7)
oos.append(evaluate(test.home_win, pL, name='LOOKAHEAD (full-season stats)', show_calibration=False))

print(compare(oos).to_string(index=False))


### Verdict: the edge was leakage

| config | Brier | log loss | acc | ECE |
|---|---|---|---|---|
| LOOKAHEAD (full-season stats) | 0.231 | 0.653 | 58.9% | 0.061 |
| **always base-rate (0.559)** | **0.247** | 0.686 | 55.9% | 0.000 |
| always 0.50 | 0.250 | 0.693 | 55.9% | 0.059 |
| OOS + lineups + SP (data ≤ Jul 1) | 0.253 | 0.699 | 54.2% | 0.100 |
| OOS team-only (data ≤ Jul 1) | 0.254 | 0.700 | 53.4% | 0.086 |
| OOS + starter ERA (data ≤ Jul 1) | 0.255 | 0.704 | 57.0% | 0.114 |

**Every honest config is worse than a coin flip (0.250) and clearly worse than
just predicting the home base rate (0.247).** The lookahead version scores 0.231
on the *same* August games — a 0.022 Brier gap that was entirely the model
grading its own homework. The earlier +18% backtest ROI came from that gap.

Why it fails out of sample:

* **Overconfident.** ECE 0.10–0.11 vs 0.059 for a coin — it makes strong calls
  that are wrong more often than a naive guess. Half-season team/hitter rates are
  noisy and the Log5 + compounding sim turns that noise into confident errors.
* **No home-field advantage.** August home teams won 55.9%; the model averages
  ~50%. That single miscalibration is why 'always base-rate' wins.
* **Starter ERA doesn't carry forward.** It lifts OOS *accuracy* to 57% (it does
  rank games slightly) but wrecks Brier and calibration — a pitcher's
  half-season ERA barely predicts his next month.

What would have to change before this is worth betting:

1. **Home-field term** — the cheapest, biggest calibration fix.
2. **Regress inputs hard or use projections** — rest-of-season pitcher/hitter
   projections instead of raw half-season rates.
3. **Fit a calibration map** (Platt / isotonic) on a train period so the output
   probabilities stop being overconfident.
4. **Reset the target:** *matching* the market (Brier ~0.245) with honest inputs
   is the realistic goal. Beating it by enough to clear ~4.5% fees is a much
   harder problem — which is the normal reason sports models don't print money.


# Model 1f — home-field term + Markov-sim review

`predict_games(..., home_field=h)` multiplies the home team's offense rates by
`(1+h)` and the away team's by `(1-h)`. Default `0.024`, calibrated so the mean
P(home win) lands near the long-run MLB rate of ~0.54. `home_field=0` disables it.


In [ ]:
# OOS (data <= Jul 1 -> August), sweeping the home-field term
rows = [
    evaluate(test.home_win, np.full(len(test), 0.5), name='always 0.50', show_calibration=False),
    evaluate(test.home_win, np.full(len(test), test.home_win.mean()),
             name='always base-rate', show_calibration=False),
]
for hfa in [0.0, 0.024, 0.035]:
    p = mc.predict_games(test, tab_cut, home_field=hfa, n=60_000, seed=7)
    rows.append(evaluate(test.home_win, p, name=f'OOS team-only  hfa={hfa}', show_calibration=False))
for hfa in [0.0, 0.024]:
    p = mc.predict_games(test, tab_cut, lineups=lu_aug, player_rates=pr_cut,
                         starters=st_cut, starter_strength=0.30, home_field=hfa, n=60_000, seed=7)
    rows.append(evaluate(test.home_win, p, name=f'OOS lineups+SP  hfa={hfa}', show_calibration=False))
print(compare(rows).to_string(index=False))


### Home-field results (August OOS)

| config | Brier | log loss | acc | ECE |
|---|---|---|---|---|
| always base-rate (0.559) | **0.2466** | 0.686 | 55.9% | 0.000 |
| OOS lineups+SP, hfa 0.024 | 0.2488 | 0.693 | 55.6% | 0.088 |
| always 0.50 | 0.2500 | 0.693 | 55.9% | 0.059 |
| OOS team-only, hfa 0.024 | 0.2500 | 0.693 | 54.4% | 0.068 |
| OOS lineups+SP, **hfa 0** | 0.2525 | 0.699 | 54.4% | 0.106 |
| OOS team-only, **hfa 0** | 0.2536 | 0.700 | 53.4% | 0.086 |

The home-field term is a real, correct fix: OOS Brier drops ~0.004–0.005 and ECE
falls from ~0.09–0.11 toward ~0.07–0.09. It takes the honest model from *worse
than a coin* to *about a coin*.

**It is still not enough.** The honest OOS model does not beat 'always predict
55.9%' (Brier 0.247). Calibration is still loose. HFA removed the obvious bias;
the rest of the gap is model quality.


### Markov / inning-sim review

**Bug fixed (minor):** `np.searchsorted(cdf, u)` could return index 6 when the
probability vector summed to 0.999… in floating point, producing a no-op step.
Clamped to `[0, 5]`. Effect was negligible (a rare wasted iteration).

**Everything correct:** event sampling, the dead-trial mask (finished trials are
frozen, not re-counted), HBP folded into BB (identical base behaviour), the
P(home win) algebra — `Σ h9[i]·P(away < i)` for P(home>away), the tie term, and the
extra-innings split are right. The convolution trick (score = 9 × half-inning pmf)
is a sound, low-variance shortcut.

**Modelling gaps, roughly by size of effect on P(home win):**

1. **Park factors — not modelled at all.** Coors vs a pitchers' park is a bigger
   per-game swing than home-field. Team rates aren't park-adjusted and the game is
   played in the home park. This is likely the #1 remaining source of game-to-game
   error. Fix: scale event rates by the home park's HR/hit factors.
2. **Baserunning is off in shape.** Single always scores runners from 2nd *and*
   3rd (too aggressive; real ~60% from 2nd). Double never scores the runner from
   1st (too conservative; real ~40%). No GIDP (inflates offense), no sac fly, no
   steals, no reached-on-error. Aggregate R/9 lands ~4.1 vs 4.3 so errors partly
   cancel, but the run *distribution* is wrong.
3. **One pitching blob for all 9 innings.** No times-through-the-order penalty
   (starters ~0.3 R worse 3rd time through), no bullpen (relievers ~0.4 R/9
   better). Innings 7–9 use team-season pitching — the two errors partly offset.
4. **Extra innings** use the pre-2020 rule (both bat a full inning until someone
   leads). Real MLB has the ghost runner on 2nd since 2020 — ~2× the scoring and
   a small extra edge to the home team (bats last). ~8% of games.
5. **No platoon/handedness, innings assumed i.i.d.** (no fatigue, no lineup-turn
   effects, no leverage-based bullpen use).

**Priority to improve discrimination:** park factors > single/double advancement +
GIDP > bullpen/TTO split > extra-innings ghost runner.
